In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler

In [ ]:
# Example: load dataset (replace with your own)
from sklearn.datasets import load_iris
data = load_iris()
X = data.data
y = data.target

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Step 1: Train KNN with different distance metrics
metrics = ['euclidean', 'manhattan', 'chebyshev']
results = pd.DataFrame(columns=['Metric', 'Accuracy', 'Precision', 'Recall'])

for metric in metrics:
    knn = KNeighborsClassifier(n_neighbors=5, metric=metric)
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    results = results.append({
        'Metric': metric,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, average='weighted'),
        'Recall': recall_score(y_test, y_pred, average='weighted')
    }, ignore_index=True)

print("KNN with Different Metrics:")
results

In [ ]:
# Step 2: KNN without feature scaling
knn_no_scaling = KNeighborsClassifier(n_neighbors=5)
knn_no_scaling.fit(X_train, y_train)
y_pred_no_scaling = knn_no_scaling.predict(X_test)

acc = accuracy_score(y_test, y_pred_no_scaling)
prec = precision_score(y_test, y_pred_no_scaling, average='weighted')
rec = recall_score(y_test, y_pred_no_scaling, average='weighted')
cm = confusion_matrix(y_test, y_pred_no_scaling)

print("KNN without scaling:")
print(f"Accuracy: {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall: {rec:.4f}")
print("Confusion Matrix:")
print(cm)

In [ ]:
# Optional: Scale features for better KNN performance
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Step 3: k-fold cross-validation to find optimal k
k_values = list(range(1, 31))
cv_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())

# Plot k vs mean CV accuracy
plt.figure(figsize=(10,6))
plt.plot(k_values, cv_scores, marker='o')
plt.xlabel('Number of Neighbors k')
plt.ylabel('Mean 5-Fold CV Accuracy')
plt.title('KNN: k vs Cross-Validation Accuracy')
plt.grid(True)
plt.show()

In [ ]:
# Step 4: Identify optimal k
optimal_k = k_values[np.argmax(cv_scores)]
print(f"Optimal k: {optimal_k}")

In [ ]:
# Step 5: Train final KNN with optimal k
final_knn = KNeighborsClassifier(n_neighbors=optimal_k)
final_knn.fit(X_train_scaled, y_train)
y_final_pred = final_knn.predict(X_test_scaled)

# Performance
final_acc = accuracy_score(y_test, y_final_pred)
final_prec = precision_score(y_test, y_final_pred, average='weighted')
final_rec = recall_score(y_test, y_final_pred, average='weighted')
final_cm = confusion_matrix(y_test, y_final_pred)

print(f"Final KNN Performance (k={optimal_k}):")
print(f"Accuracy: {final_acc:.4f}")
print(f"Precision: {final_prec:.4f}")
print(f"Recall: {final_rec:.4f}")
print("Confusion Matrix:")
print(final_cm)

In [ ]:
# Step 6: Combine all results into a DataFrame for comparison
comparison_df = results.append({
    'Metric': f'No Scaling k=5',
    'Accuracy': acc,
    'Precision': prec,
    'Recall': rec
}, ignore_index=True)

comparison_df = comparison_df.append({
    'Metric': f'Optimal k={optimal_k}',
    'Accuracy': final_acc,
    'Precision': final_prec,
    'Recall': final_rec
}, ignore_index=True)

print("\nComparison of KNN Results:")
comparison_df